# CASMI 2026 · Yash · Model 1: analog + modification enumeration (E711, gate 0.7)

Generates new candidate structures (Class 3) by applying one small edit (±O, ±CH2, ±H2, ±hexose) to library molecules whose mass is exactly one edit away from the query. Library candidates keep the baseline cosine; generated products get their parent's cosine, scaled to 0.3 when the best library match already has cosine ≥ 0.7.

Inputs: competition data + `casmi-team-src`. Internet OFF, CPU only, no weights.

In [ ]:
import glob, os, sys, time
t0 = time.time()

# Team code (src/) from the casmi-team-src dataset
SRC = glob.glob("/kaggle/input/**/casmi_src", recursive=True)[0]
sys.path.insert(0, SRC)

# RDKit is needed for fingerprints. If it's missing, install it from the team wheels dataset.
try:
    import rdkit
except ImportError:
    os.system("pip install --no-index --find-links /kaggle/input/casmi-wheels rdkit -q")

from src.data.build_reference import build
from src.evaluation.validate import make_submission
from src.models.base import Scorer, CosineScorer

TEST = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
DATA = os.path.dirname(TEST)
REF = "/kaggle/working/ref"
build(os.path.join(DATA, "train.parquet"), REF)   # ~3-5 min
print(f"reference built in {time.time() - t0:.0f}s")

## Model (copied from `members/YashShevkar30/model_analog_enum.py`)

In [ ]:
import argparse
import json
import os
import time

import numpy as np

from src.data.spectra import cosine, neutral_mass
from src.evaluation.metric import smiles_to_ik14
from src.models.base import Scorer

VAL_LIB = "enveda-np-examples"
HEXOSE = "C1OC(CO)C(O)C(O)C1O"

# name -> (mass change product - parent in Da, list of reaction SMARTS parent>>product)
MODIFICATIONS = {
    "hydroxylation": (15.994915, ["[cH:1]>>[c:1]O"]),
    "methylation": (14.015650, ["[OX2H1:1]>>[O:1]C",
                                "[NX3;H1,H2;!$(N-[!#6;!#1]):1]>>[N:1]C"]),
    "demethylation": (-14.015650, ["[O:1]-[CH3]>>[O:1]", "[N:1]-[CH3]>>[N:1]"]),
    "hydrogenation": (2.015650, ["[C:1]=[C:2]>>[C:1]-[C:2]"]),
    "dehydrogenation": (-2.015650, ["[CX4;!H0:1]-[CX4;!H0:2]>>[C:1]=[C:2]"]),
    "glycosylation": (162.052824, ["[OX2H1:1]>>[O:1]" + HEXOSE]),
    "deglycosylation": (-162.052824, [
        "[*:2]-[O:1]-[CH1;R]1-O-[CH1](-[CH2]-[OH])-[CH1](-[OH])-[CH1](-[OH])-[CH1]1-[OH]>>[*:2]-[O:1]"]),
}


def query_mass(spectra):
    masses = [neutral_mass(s["precursor_mz"], s["adduct"]) for s in spectra]
    masses = [m for m in masses if np.isfinite(m)]
    return float(np.median(masses)) if masses else np.nan


def spectral_score(lib, spectra, struct_idx, max_ref=20):
    """Same rule as the baseline: mean over query spectra of the best cosine vs the structure's
    reference spectra in the same ionisation mode."""
    refs = list(lib.spectra_of(struct_idx))[:max_ref]
    per_query = []
    for q in spectra:
        best = 0.0
        for j in refs:
            if lib.mode[j] != q["ionization_mode"]:
                continue
            mz_r, in_r = lib.peaks(j)
            best = max(best, cosine(q["mz"], q["intensity"], mz_r, in_r))
        per_query.append(best)
    return float(np.mean(per_query)) if per_query else 0.0


class AnalogEnumScorer(Scorer):
    name = "analog-enum"

    def __init__(self, ref_dir="data/processed", exclude_libs=(VAL_LIB,), ppm=15.0,
                 analogs_per_mod=3, max_products=200, gen_weight=1.0, mods=None,
                 gate=None, gated_weight=0.3):
        from rdkit import RDLogger
        from rdkit.Chem import AllChem

        from src.models.library_search import ReferenceLibrary

        RDLogger.DisableLog("rdApp.*")
        # Validation: exclude the validation library, exactly like src.evaluation.validate does.
        # Kaggle: pass exclude_libs=() to use every library.
        self.lib = ReferenceLibrary(ref_dir, exclude_libs=list(exclude_libs))
        self.ppm = ppm
        self.analogs_per_mod = analogs_per_mod
        self.max_products = max_products
        self.gen_weight = gen_weight
        # Confidence gate: if the best library candidate already matches well (cosine >= gate),
        # the molecule is probably in the library, so generated products get gated_weight instead.
        self.gate = gate
        self.gated_weight = gated_weight
        mods = mods or list(MODIFICATIONS)
        self.rxns = {m: (MODIFICATIONS[m][0], [AllChem.ReactionFromSmarts(s) for s in MODIFICATIONS[m][1]])
                     for m in mods}
        self._gen = {}      # inchikey14 -> score for the current molecule's products
        self.stats = []     # one dict per molecule, aligned with evaluate()'s molecule order

    # ---- generation ----
    def _apply(self, rxns, parent_smiles):
        from rdkit import Chem
        mol = Chem.MolFromSmiles(parent_smiles)
        if mol is None:
            return [], 0
        out, n_raw = [], 0
        for rx in rxns:
            for prods in rx.RunReactants((mol,)):
                n_raw += 1
                p = Chem.Mol(prods[0])
                try:
                    Chem.SanitizeMol(p)
                except Exception:
                    continue
                out.append(p)
        return out, n_raw

    def extra_candidates(self, spectra):
        from rdkit import Chem
        from rdkit.Chem.rdMolDescriptors import CalcExactMolWt

        t0 = time.time()
        self._gen = {}
        st = {"n_parents": 0, "n_raw": 0, "n_valid": 0, "n_in_mass": 0, "n_unique": 0}
        M = query_mass(spectra)
        if not np.isfinite(M):
            self.stats.append({**st, "keys": [], "seconds": 0.0})
            return []
        best = {}   # inchikey14 -> (score, smiles, mod)
        for mod, (delta, rxns) in self.rxns.items():
            parents = self.lib.candidates(M - delta, self.ppm)
            if parents.size == 0:
                continue
            scored = sorted(((spectral_score(self.lib, spectra, s), s) for s in parents), reverse=True)
            for p_score, s in scored[: self.analogs_per_mod]:
                st["n_parents"] += 1
                prods, n_raw = self._apply(rxns, self.lib.structs.smiles.values[s])
                st["n_raw"] += n_raw
                st["n_valid"] += len(prods)
                for p in prods:
                    if abs(CalcExactMolWt(p) - M) > M * self.ppm * 1e-6:
                        continue
                    st["n_in_mass"] += 1
                    smi = Chem.MolToSmiles(p)
                    key = smiles_to_ik14(smi, tautomer=False)  # same key validate.py dedupes on
                    if key and (key not in best or p_score > best[key][0]):
                        best[key] = (p_score, smi, mod)
        ranked = sorted(best.items(), key=lambda kv: -kv[1][0])[: self.max_products]
        self._gen = {k: v[0] for k, v in ranked}
        st["n_unique"] = len(ranked)
        self.stats.append({**st, "keys": [k for k, _ in ranked], "mods": [v[2] for _, v in ranked],
                           "seconds": time.time() - t0})
        return [v[1] for _, v in ranked]

    # ---- scoring ----
    def score(self, spectra, candidates):
        scores = candidates.cosine.values.astype(float).copy()
        gen = candidates.struct_idx.values == -1
        w = self.gen_weight
        if self.gate is not None and (~gen).any() and scores[~gen].max() >= self.gate:
            w = self.gated_weight
        scores[gen] = [w * self._gen.get(k, 0.0) for k in candidates.inchikey14.values[gen]]
        return scores


def summarise(res, model):
    """Generation statistics for the report, joined to evaluate()'s per-molecule table."""
    per = res["per_molecule"].copy()
    st = model.stats[: len(per)]
    for col in ["n_parents", "n_raw", "n_valid", "n_in_mass", "n_unique", "seconds"]:
        per[col] = [s[col] for s in st]
    per["true_generated"] = [m in set(s["keys"]) for m, s in zip(per.molecule, st)]
    per["true_mod"] = [s["mods"][s["keys"].index(m)] if m in s["keys"] else None
                       for m, s in zip(per.molecule, st)]
    summary = {
        "model": res["model"], "mrr": res["mrr"], "top1": res["top1"], "top10": res["top10"],
        "top25": res["top25"], "n_molecules": len(per),
        "mean_products": float(per.n_unique.mean()),
        "valid_rate": float(per.n_valid.sum() / max(per.n_raw.sum(), 1)),
        "in_mass_rate": float(per.n_in_mass.sum() / max(per.n_valid.sum(), 1)),
        "recall_generated": float(per.true_generated.mean()),
        "true_by_mod": per.true_mod.value_counts().to_dict(),
        "sec_per_molecule": float(per.seconds.mean()),
    }
    return per, summary


In [ ]:
model = AnalogEnumScorer(ref_dir=REF, exclude_libs=(), gate=0.7)   # full library on Kaggle


In [ ]:
sub = make_submission(model, TEST, REF, "/kaggle/working/submission.csv")
for f in os.listdir(REF):          # keep only submission.csv in the output
    os.remove(os.path.join(REF, f))
os.rmdir(REF)
print(sub.shape, f"total {time.time() - t0:.0f}s")
sub.head()